In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. **Install Ollama** from [https://ollama.com/download](https://ollama.com/download)\n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start Ollama locally** by opening a terminal and running:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\n3. **Test the local server** with:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response like:\n   ```json\n   {"models": [...]}  \n   ```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'I can help, but I need a bit more context. Which course are you referring to, and do you mean:\n\n- joining as a student,\n- enrolling after it has started, or\n- getting access to materials only?\n\nIf you want, I can also help you draft a short message to the instructor asking whether late enrollment is possible.'

## Tool

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

Response(id='resp_0a0517925cbedb2d006a3227ff253481989424a2c0fb5a013f', created_at=1781671935.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0a0517925cbedb2d006a3227ffbb5c8198a2f11606a8c9e66d', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf your goal is to get a certificate, make sure you submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ database 

In [ ]:
import json

call = response.output[0]
args = json.loads(call.arguments) # type: ignore

results = search(**args)
result_json = json.dumps(results, indent=2)

In [10]:
messages.extend(response.output)

In [11]:
messages.append({
  "type": "function_call_output",
  "call_id": call.call_id, # type: ignore
  "output": result_json,
})

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join and start learning now.

If your goal is to get a certificate, you’ll need to submit your project while the course is still accepting submissions.


## Usage Cost

In [ ]:
usage = response.usage
usage.input_tokens, usage.output_tokens # type: ignore

(771, 39)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


## Function-call Helper

In [51]:
def make_call(call):
  args = json.loads(call.arguments)

  if call.name == "search":
    result = search(**args)

  result_json = json.dumps(result, indent=2)

  return {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
  }

In [52]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [53]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run local install setup"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. Install it from [https://ollama.com/download](https://ollama.com/download)
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model in your terminal:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test the local server:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python, install the client:
   ```bash
   pip install ollama
   ```

If you want, I can also show the minimal Python example for calling Ollama.


'To run Ollama locally:\n\n1. Install it from [https://ollama.com/download](https://ollama.com/download)\n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a model in your terminal:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\n3. To test the local server:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. If you want to use it from Python, install the client:\n   ```bash\n   pip install ollama\n   ```\n\nIf you want, I can also show the minimal Python example for calling Ollama.'

In [58]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I join it FAQ"}
iteration #2...
function_call: search {"query":"certificate project accepting submissions while we're still accepting submissions self-paced live cohort join course discovered course FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you’re just learning, you can start anytime and work through the materials at your own pace.

If you want, I can also help with how to start the course or explain the certificate requirements.


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. If you’re just learning, you can start anytime and work through the materials at your own pace.\n\nIf you want, I can also help with how to start the course or explain the certificate requirements.'

In [64]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "What's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"chess opening queen gambit"}
iteration #3...
ASSISTANT:
I couldn’t find a relevant course FAQ entry for “queen gambit,” so this may be off-topic for the course.

If you meant **Queen’s Gambit** in chess, it’s a chess opening where White starts with:
1. d4 d5
2. c4

If you want, I can also help with course-related questions—are there other areas you want to explore?


'I couldn’t find a relevant course FAQ entry for “queen gambit,” so this may be off-topic for the course.\n\nIf you meant **Queen’s Gambit** in chess, it’s a chess opening where White starts with:\n1. d4 d5\n2. c4\n\nIf you want, I can also help with course-related questions—are there other areas you want to explore?'